# Conflict Early Warning Nigeria — Phase 1
## Data Exploration, Audit & Naive Baseline

**Goal:** Understand the ACLED weekly aggregated Nigeria dataset before fitting any model.  
Establish the naive baseline (last observed month = forecast) that every future model must beat.  
Produce a written data-quality audit flagging coverage gaps.

**Target variable we will predict throughout the project:**  
> `P(at least 1 fatal event in state X in month M+1)` — a binary probability between 0 and 1.

---

### How to run
```bash
pip install -r ../requirements.txt
jupyter nbconvert --to notebook --execute 01_eda_baseline.ipynb --output 01_eda_baseline_executed.ipynb
```

---

## 0 · Setup and data loading

In [ ]:
# ─── Standard imports ─────────────────────────────────────────────────────────
import json, warnings, pathlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

# ─── Paths ────────────────────────────────────────────────────────────────────
ROOT        = pathlib.Path('..').resolve()
RAW_DIR     = ROOT / 'data' / 'raw'
PROC_DIR    = ROOT / 'data' / 'processed'
METRICS_DIR = ROOT / 'metrics'
REPORTS_DIR = ROOT / 'reports'
for d in [PROC_DIR, METRICS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Expected column schema for the ACLED weekly aggregated CSV ───────────────
#   week             : str  'YYYY-MM-DD' (Monday of the ISO week)
#   admin1           : str  Nigerian state name (36 states + FCT)
#   event_type       : str  ACLED primary category (e.g. 'Battles')
#   sub_event_type   : str  ACLED secondary category
#   events           : int  Number of distinct events
#   fatalities       : int  Number of fatalities (may be 0)
#   latitude         : float  Centroid of admin1 (NOT event location)
#   longitude        : float
EXPECTED_COLS = [
    'week', 'admin1', 'event_type', 'sub_event_type',
    'events', 'fatalities', 'latitude', 'longitude'
]

print('Setup complete.')
print(f'Data will be read from: {RAW_DIR}')

In [ ]:
# ─── Load data ────────────────────────────────────────────────────────────────
# ASSUMPTION: your ACLED export is saved as:
#   data/raw/acled_nigeria_weekly_agg.csv
# If you used a different filename, change this path.

DATA_FILE = RAW_DIR / 'acled_nigeria_weekly_agg.csv'

# ── DEMO MODE ─────────────────────────────────────────────────────────────────
# If the file does not exist yet, we synthesise a realistic stub so every
# cell below runs without error. The stub mimics the real column schema.
# Replace with the real file when available.
DEMO_MODE = not DATA_FILE.exists()

if DEMO_MODE:
    print('⚠️  Real data file not found — running in DEMO MODE with synthetic data.')
    print(f'   Expected path: {DATA_FILE}')
    print('   Save your ACLED export there and re-run.\n')

    rng = np.random.default_rng(42)

    STATES = [
        'Zamfara', 'Katsina', 'Sokoto', 'Kaduna',           # Northwest pilot
        'Borno', 'Yobe', 'Adamawa',                          # Northeast
        'Lagos', 'Ogun', 'Oyo',                              # Southwest
        'Rivers', 'Delta', 'Bayelsa',                        # South-South
        'Plateau', 'Benue', 'Nasarawa',                      # North-Central
        'Kano', 'Jigawa', 'Kebbi',                           # Northwest cont.
        'Niger', 'Kwara', 'FCT',                             # North-Central cont.
    ]
    STATE_COORDS = {
        'Zamfara': (12.17, 6.66), 'Katsina': (12.99, 7.60),
        'Sokoto': (13.06, 5.24), 'Kaduna': (10.52, 7.44),
        'Borno': (11.85, 13.16), 'Yobe': (12.29, 11.44),
        'Adamawa': (9.33, 12.39), 'Lagos': (6.52, 3.38),
        'Ogun': (6.99, 3.47), 'Oyo': (7.85, 3.93),
        'Rivers': (4.85, 7.04), 'Delta': (5.52, 5.90),
        'Bayelsa': (4.77, 6.07), 'Plateau': (9.22, 9.52),
        'Benue': (7.34, 8.74), 'Nasarawa': (8.49, 8.52),
        'Kano': (12.00, 8.52), 'Jigawa': (12.21, 9.56),
        'Kebbi': (11.49, 4.20), 'Niger': (10.00, 5.60),
        'Kwara': (8.49, 4.55), 'FCT': (9.07, 7.40),
    }
    EVENT_TYPES = ['Battles', 'Violence against civilians',
                   'Explosions/Remote violence', 'Protests', 'Riots',
                   'Strategic developments']
    SUB_TYPES = {
        'Battles': ['Armed clash', 'Government regains territory', 'Non-state actor overtakes territory'],
        'Violence against civilians': ['Attack', 'Abduction/forced disappearance', 'Sexual violence'],
        'Explosions/Remote violence': ['IED', 'Air/drone strike', 'Grenade'],
        'Protests': ['Peaceful protest', 'Excessive force against protesters'],
        'Riots': ['Mob violence', 'Violent demonstration'],
        'Strategic developments': ['Agreement', 'Looting/property destruction', 'Arrests'],
    }

    weeks = pd.date_range('2015-01-05', '2024-12-30', freq='W-MON')
    rows = []
    for w in weeks:
        for state in STATES:
            for et in rng.choice(EVENT_TYPES, size=rng.integers(0, 4), replace=False):
                # Higher conflict rate in Northwest and Northeast
                intensity = 3 if state in ['Zamfara','Katsina','Sokoto','Kaduna','Borno','Yobe'] else 0.5
                n_events = int(rng.negative_binomial(1, 1/(1+intensity)))
                if n_events == 0:
                    continue
                fatal_rate = 0.6 if state in ['Zamfara','Borno'] else 0.3
                n_fatal = int(rng.negative_binomial(n_events, 1/(1+fatal_rate*intensity)))
                lat, lon = STATE_COORDS.get(state, (9.0, 8.0))
                sub = rng.choice(SUB_TYPES[et])
                rows.append({
                    'week': w.strftime('%Y-%m-%d'), 'admin1': state,
                    'event_type': et, 'sub_event_type': sub,
                    'events': n_events, 'fatalities': n_fatal,
                    'latitude': round(lat + rng.normal(0, 0.01), 4),
                    'longitude': round(lon + rng.normal(0, 0.01), 4),
                })
    df_raw = pd.DataFrame(rows)
else:
    print(f'Loading real data from {DATA_FILE}')
    df_raw = pd.read_csv(DATA_FILE)

print(f'Rows loaded: {len(df_raw):,}')
print(f'Columns   : {list(df_raw.columns)}')
df_raw.head(3)

## 1 · Schema validation

In [ ]:
# ─── 1a. Column check ─────────────────────────────────────────────────────────
# PURPOSE: catch schema drift early — ACLED sometimes renames columns
actual_cols = set(df_raw.columns)
expected_cols = set(EXPECTED_COLS)

missing = expected_cols - actual_cols
extra   = actual_cols - expected_cols

print('=== Schema check ===')
print(f'Missing expected columns : {missing if missing else "None ✓"}')
print(f'Unexpected extra columns : {extra if extra else "None ✓"}')

# If columns are missing, map them here before proceeding
# e.g. df_raw.rename(columns={'region': 'admin1'}, inplace=True)

# ─── 1b. Type coercion ────────────────────────────────────────────────────────
df_raw['week']       = pd.to_datetime(df_raw['week'])
df_raw['events']     = pd.to_numeric(df_raw['events'],     errors='coerce').astype('Int64')
df_raw['fatalities'] = pd.to_numeric(df_raw['fatalities'], errors='coerce').astype('Int64')
df_raw['latitude']   = pd.to_numeric(df_raw['latitude'],   errors='coerce')
df_raw['longitude']  = pd.to_numeric(df_raw['longitude'],  errors='coerce')
df_raw['admin1']     = df_raw['admin1'].str.strip().str.title()

# ─── 1c. Null audit ───────────────────────────────────────────────────────────
null_counts = df_raw.isnull().sum()
print('\n=== Null counts ===')
print(null_counts[null_counts > 0].to_string() or 'No nulls ✓')

# ─── 1d. Negative values — sanity check ──────────────────────────────────────
print('\n=== Negative counts check ===')
print(f'events     < 0: {(df_raw["events"]     < 0).sum()}')
print(f'fatalities < 0: {(df_raw["fatalities"] < 0).sum()}')
print(f'\nDate range: {df_raw["week"].min().date()} → {df_raw["week"].max().date()}')
print(f'States ({df_raw["admin1"].nunique()}): {sorted(df_raw["admin1"].unique())}')

## 2 · Temporal coverage — are there silent weeks?

In [ ]:
# PURPOSE: Weeks with zero rows for a state could mean (a) genuine quiet
# or (b) no coder was assigned that week — both look the same in the data.
# We flag these explicitly rather than imputing zeros blindly.

all_weeks  = pd.date_range(df_raw['week'].min(), df_raw['week'].max(), freq='W-MON')
all_states = df_raw['admin1'].unique()

# Build complete grid: every (week, state) combination should appear
full_grid = pd.MultiIndex.from_product(
    [all_weeks, all_states], names=['week', 'admin1']
).to_frame(index=False)

# Weeks present per state (collapsing event_type dimension)
weeks_present = df_raw.groupby(['admin1','week']).size().reset_index(name='rows')
coverage = full_grid.merge(weeks_present, on=['week','admin1'], how='left')
coverage['is_silent'] = coverage['rows'].isna()

silent_pct = coverage.groupby('admin1')['is_silent'].mean().sort_values(ascending=False)
print('=== Fraction of weeks silent (zero rows in data) per state ===')
print(silent_pct.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
silent_pct.sort_values().plot.barh(ax=ax, color='#c0392b')
ax.set_xlabel('Fraction of weeks with no rows', fontsize=11)
ax.set_title('Coverage gaps by state\n(silent week = no rows in ACLED export)', fontsize=12)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig01_silent_weeks_by_state.png', dpi=150)
plt.show()

# ⚠️ What to check in the output:
# States with > 20% silent weeks — especially if they are in the Northwest pilot —
# need a 'LOW COVERAGE' flag in every subsequent analysis.

## 3 · Event-type breakdown

In [ ]:
# PURPOSE: Understand what kinds of violence dominate.
# Battles and Violence against civilians are most relevant to the NGO use case.

et_summary = (
    df_raw.groupby('event_type')[['events','fatalities']]
    .sum()
    .sort_values('fatalities', ascending=False)
    .reset_index()
)
et_summary['fatality_share'] = et_summary['fatalities'] / et_summary['fatalities'].sum()
print(et_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
et_summary.set_index('event_type')['events'].sort_values().plot.barh(ax=axes[0], color='#2980b9')
axes[0].set_title('Total events by type')
et_summary.set_index('event_type')['fatalities'].sort_values().plot.barh(ax=axes[1], color='#c0392b')
axes[1].set_title('Total fatalities by type')
for ax in axes:
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig02_event_type_breakdown.png', dpi=150)
plt.show()

# ⚠️ What to check:
# If 'Protests' or 'Strategic developments' dominate fatalities — that is suspicious;
# it may indicate a classification error in the source data.

## 4 · Time series by state — Northwest pilot focus

In [ ]:
# PURPOSE: Visualise the raw time series for the four pilot states.
# These charts will be the visual anchor for all model outputs later.

PILOT_STATES = ['Zamfara', 'Katsina', 'Sokoto', 'Kaduna']

# Aggregate to weekly totals across event types for the pilot states
weekly_pilot = (
    df_raw[df_raw['admin1'].isin(PILOT_STATES)]
    .groupby(['week', 'admin1'])[['events','fatalities']]
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(4, 2, figsize=(15, 14), sharex=True)
for i, state in enumerate(PILOT_STATES):
    d = weekly_pilot[weekly_pilot['admin1'] == state].sort_values('week')
    axes[i, 0].bar(d['week'], d['events'], width=6, color='#2980b9', alpha=0.7)
    axes[i, 0].set_title(f'{state} — events/week')
    axes[i, 1].bar(d['week'], d['fatalities'], width=6, color='#c0392b', alpha=0.7)
    axes[i, 1].set_title(f'{state} — fatalities/week')
    for ax in [axes[i,0], axes[i,1]]:
        ax.tick_params(axis='x', rotation=30)
plt.suptitle('Northwest Nigeria pilot states — weekly conflict intensity', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig03_pilot_states_timeseries.png', dpi=150)
plt.show()

# ⚠️ What to check:
# Long flat stretches (all zeros) in a state with known conflict
# = coverage gap. Note the approximate date ranges.

## 5 · Collapse to monthly admin1 panel — the modelling unit

In [ ]:
# RATIONALE: Weekly data has many zeros (sparse) and high week-to-week noise.
# Monthly panels give enough events per cell for the negative-binomial model,
# and monthly forecasts are more useful to NGO planners.
# We keep weekly data in data/processed/ for fine-grained analyses later.

df_raw['month'] = df_raw['week'].dt.to_period('M')  # e.g. '2023-04'

monthly = (
    df_raw.groupby(['month', 'admin1'])
    .agg(
        events=('events', 'sum'),
        fatalities=('fatalities', 'sum'),
        weeks_observed=('week', 'nunique'),   # how many weekly rows we had
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first'),
    )
    .reset_index()
)

# Derive the target variable
# `fatal_event` = 1 if at least one fatality was recorded that month in that state
monthly['fatal_event'] = (monthly['fatalities'] > 0).astype(int)

# Coverage quality: we expect 4–5 weeks per month.
# If weeks_observed < 3, flag the cell as low quality.
monthly['low_coverage'] = (monthly['weeks_observed'] < 3).astype(int)

print(f'Monthly panel shape: {monthly.shape}')
print(f'\nDate range (months): {monthly["month"].min()} → {monthly["month"].max()}')
print(f'States: {monthly["admin1"].nunique()}')
print(f'\nTarget prevalence (fatal_event = 1): {monthly["fatal_event"].mean():.2%}')
print(f'Low-coverage cells:                   {monthly["low_coverage"].mean():.2%}')

monthly.head()

In [ ]:
# Save processed panel
out_path = PROC_DIR / 'monthly_admin1_panel.parquet'
monthly.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')

## 6 · Descriptive statistics and distribution check

In [ ]:
# PURPOSE: Understand the count distributions. Conflict data is almost always
# overdispersed (variance >> mean), which rules out Poisson and motivates
# the negative-binomial model in Phase 4.

for col in ['events', 'fatalities']:
    s = monthly[col].dropna()
    mean, var = s.mean(), s.var()
    print(f'{col:12s}: mean={mean:.2f}, var={var:.2f}, '
          f'var/mean={var/mean:.2f} '
          f'[>1 = overdispersed → use negative binomial]')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(axes, ['events','fatalities'], ['#2980b9','#c0392b']):
    cap = monthly[col].quantile(0.99)  # cap at 99th pct to avoid giant right tails
    ax.hist(monthly[col].clip(upper=cap), bins=50, color=color, alpha=0.8)
    ax.set_title(f'Monthly {col} distribution\n(capped at 99th pct = {cap:.0f})')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig04_count_distributions.png', dpi=150)
plt.show()

# ⚠️ What to check:
# If var/mean >> 1 (e.g., >5) a Poisson model will dramatically underestimate
# spike events. We already plan for negative-binomial — this confirms it.

## 7 · Coverage-bias analysis — 'quiet' ≠ 'peaceful'

In [ ]:
# EXPLANATION: In conflict research, areas with few journalists, poor roads, or
# active fighting are systematically underrepresented in event databases.
# A 'quiet' record may mean no events, OR no reporters to observe them.
# We flag this structurally using:
#   - low weeks_observed (we already computed this)
#   - states with suspiciously long zero-fatality streaks relative to neighbours

def zero_streak(series: pd.Series) -> int:
    """Length of current trailing zero streak in a sorted series."""
    reversed_series = series.iloc[::-1].values
    streak = 0
    for v in reversed_series:
        if v == 0:
            streak += 1
        else:
            break
    return streak

streak_by_state = (
    monthly.sort_values('month')
    .groupby('admin1')['fatalities']
    .apply(zero_streak)
    .sort_values(ascending=False)
)
print('=== Current trailing zero-fatality streak (months) per state ===')
print(streak_by_state.to_string())

fig, ax = plt.subplots(figsize=(10, 6))
streak_by_state.sort_values().plot.barh(ax=ax, color='#e67e22')
ax.axvline(6, color='red', linestyle='--', label='6-month threshold')
ax.set_xlabel('Months of zero recorded fatalities (most recent)')
ax.set_title('States with recent zero-fatality streaks\n'
             'Streaks > 6 months in states near active conflict zones = coverage risk')
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig05_zero_fatality_streaks.png', dpi=150)
plt.show()

In [ ]:
# ─── Coverage-bias flags (written to audit report) ────────────────────────────
# Rules:
#  FLAG 1: silent week fraction > 30% for any pilot state
#  FLAG 2: zero-fatality streak > 6 months for a state bordering a hotspot
#  FLAG 3: weeks_observed < 3 for any month in pilot states

BORDERING_HOTSPOT = ['Zamfara','Katsina','Sokoto','Kaduna',
                     'Niger','Kebbi','Kano','Jigawa',
                     'Borno','Yobe','Adamawa','Plateau','Benue']

flag1 = silent_pct[silent_pct > 0.30].reset_index()
flag1.columns = ['admin1', 'silent_fraction']

flag2 = streak_by_state[
    (streak_by_state > 6) &
    streak_by_state.index.isin(BORDERING_HOTSPOT)
].reset_index()
flag2.columns = ['admin1', 'zero_streak_months']

flag3 = (
    monthly[
        (monthly['admin1'].isin(PILOT_STATES)) &
        (monthly['weeks_observed'] < 3)
    ]
    .groupby('admin1')['month']
    .count()
    .reset_index()
)
flag3.columns = ['admin1', 'low_coverage_months']

print('\n⚠️  FLAG 1 — Silent week fraction > 30%:')
print(flag1.to_string(index=False) if not flag1.empty else '  None')
print('\n⚠️  FLAG 2 — Long zero-fatality streak near hotspot:')
print(flag2.to_string(index=False) if not flag2.empty else '  None')
print('\n⚠️  FLAG 3 — Pilot state months with < 3 weeks observed:')
print(flag3.to_string(index=False) if not flag3.empty else '  None')

## 8 · Naive baseline model (M0)

In [ ]:
# EXPLANATION: The naive baseline says:
#   "My forecast for month T+1 = what I observed in month T."
# For a binary probability forecast we use:
#   p_hat = 1 if fatalities > 0 last month, else 0
# (hard 0/1 — we will calibrate this in Phase 4)
#
# This is the ceiling EVERY future model must beat. If a sophisticated model
# cannot beat this, it is not adding value and should not be shipped.

# Sort and create lagged target
monthly_sorted = monthly.sort_values(['admin1','month']).copy()
monthly_sorted['fatal_event_lag1'] = (
    monthly_sorted.groupby('admin1')['fatal_event'].shift(1)
)

# Naive forecast = last period's outcome
monthly_sorted['p_naive'] = monthly_sorted['fatal_event_lag1'].fillna(0.5)
# Note: 0.5 for the very first observation per state (no prior data)

# Evaluation set: drop rows with NaN lag (first month per state)
eval_df = monthly_sorted.dropna(subset=['fatal_event_lag1']).copy()

print(f'Evaluation rows: {len(eval_df):,}')
print(f'Target prevalence in eval set: {eval_df["fatal_event"].mean():.2%}')

In [ ]:
# ─── Brier Score ──────────────────────────────────────────────────────────────
# Brier score = mean((p_forecast - outcome)^2)
# Perfect: 0.0  |  Coin-flip (p=0.5 always): 0.25  |  Naive = somewhere between
# Lower is better.

from sklearn.metrics import brier_score_loss, roc_auc_score

y_true   = eval_df['fatal_event'].values
p_naive  = eval_df['p_naive'].values

bs_naive = brier_score_loss(y_true, p_naive)
# Brier Skill Score (BSS): how much better than a no-skill (climatological) forecast
prev     = y_true.mean()
bs_clim  = brier_score_loss(y_true, np.full_like(y_true, prev, dtype=float))
bss      = 1 - bs_naive / bs_clim

# AUC-ROC (threshold-independent discrimination measure)
try:
    auc = roc_auc_score(y_true, p_naive)
except Exception:
    auc = float('nan')

print('=== Naive baseline metrics (M0) ===')
print(f'Brier score      : {bs_naive:.4f}   (lower is better; climatological baseline={bs_clim:.4f})')
print(f'Brier Skill Score: {bss:.4f}        (>0 means better than predicting prevalence always)')
print(f'AUC-ROC          : {auc:.4f}        (0.5=random, 1.0=perfect)')
print()
print('👆 These numbers are the ceiling. Record them carefully.')
print('   Phase 4 models MUST beat the Brier score on a held-out expanding window.')

In [ ]:
# ─── Calibration plot ─────────────────────────────────────────────────────────
# PURPOSE: Check whether our probability forecasts are well-calibrated.
# A well-calibrated model: events predicted at 70% probability happen ~70% of the time.
# The naive model is binary (0 or 1), so calibration is trivial — shown for reference.

from sklearn.calibration import calibration_curve

fraction_of_positives, mean_predicted = calibration_curve(
    y_true, p_naive, n_bins=5, strategy='uniform'
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0,1],[0,1], 'k--', label='Perfect calibration')
ax.plot(mean_predicted, fraction_of_positives,
        'o-', color='#2980b9', label='Naive model M0')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration plot — Naive baseline\n'
             'Points on the dashed line = perfect calibration')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig06_calibration_naive.png', dpi=150)
plt.show()

# ⚠️ What to check:
# The naive model has only two predicted values (0 and 1), so calibration will
# look jagged. Phase 4 models will produce smooth probability distributions —
# that calibration plot is where it matters most.

In [ ]:
# ─── Expanding-window prototype (Phase 4 preview) ─────────────────────────────
# We show the logic here so the structure is visible before Phase 4 builds it fully.
# Train on months 1..T, evaluate on T+1, slide forward.

def expanding_window_naive(panel: pd.DataFrame,
                            min_train_months: int = 36) -> pd.DataFrame:
    """
    For each month T after min_train_months, compute the naive forecast
    (lagged outcome) and record the true outcome. Returns a results DataFrame.

    Parameters
    ----------
    panel : pd.DataFrame  — monthly admin1 panel with columns
                            [month, admin1, fatal_event]
    min_train_months : int — minimum history before first forecast
    """
    all_months = sorted(panel['month'].unique())
    results = []
    for i, target_month in enumerate(all_months):
        if i < min_train_months:
            continue
        prev_month = all_months[i - 1]
        prev = panel[panel['month'] == prev_month][['admin1','fatal_event']].copy()
        prev.rename(columns={'fatal_event': 'p_naive'}, inplace=True)
        cur  = panel[panel['month'] == target_month][['admin1','fatal_event']].copy()
        merged = prev.merge(cur, on='admin1', how='inner')
        merged['forecast_month'] = target_month
        results.append(merged)
    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()

ew_results = expanding_window_naive(monthly_sorted[['month','admin1','fatal_event']])

if not ew_results.empty:
    bs_ew = brier_score_loss(ew_results['fatal_event'], ew_results['p_naive'])
    print(f'Expanding-window Brier score (naive): {bs_ew:.4f}')
    print(f'Expanding-window folds evaluated    : {ew_results["forecast_month"].nunique()}')
    print()
    print('This is the PRIMARY benchmark for Phase 4.')
else:
    print('Not enough months for expanding-window evaluation — need > 36 months of data.')

## 9 · Save metrics and write audit report

In [ ]:
# ─── Persist metrics ─────────────────────────────────────────────────────────
metrics = {
    'phase': 1,
    'run_date': datetime.now().isoformat(),
    'demo_mode': DEMO_MODE,
    'data_rows_raw': len(df_raw),
    'monthly_panel_rows': len(monthly),
    'states': int(monthly['admin1'].nunique()),
    'date_range': {
        'start': str(monthly['month'].min()),
        'end':   str(monthly['month'].max()),
    },
    'target_prevalence': float(eval_df['fatal_event'].mean()),
    'low_coverage_fraction': float(monthly['low_coverage'].mean()),
    'naive_baseline_M0': {
        'brier_score': round(float(bs_naive), 4),
        'brier_skill_score': round(float(bss), 4),
        'auc_roc': round(float(auc), 4) if not np.isnan(auc) else None,
        'expanding_window_brier': round(float(bs_ew), 4) if not ew_results.empty else None,
    },
    'coverage_flags': {
        'flag1_silent_gt30pct': flag1['admin1'].tolist(),
        'flag2_zero_streak_near_hotspot': flag2['admin1'].tolist() if not flag2.empty else [],
        'flag3_pilot_low_coverage_months': flag3['admin1'].tolist() if not flag3.empty else [],
    },
}

metrics_path = METRICS_DIR / 'phase1_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved: {metrics_path}')
print(json.dumps(metrics, indent=2))

In [ ]:
# ─── Write markdown audit report ─────────────────────────────────────────────
report_path = REPORTS_DIR / '01_data_audit.md'

flag1_text = '\n'.join(f'- `{r.admin1}`: {r.silent_fraction:.0%} silent weeks' for _, r in flag1.iterrows()) or '- None identified'
flag2_text = '\n'.join(f'- `{r.admin1}`: {r.zero_streak_months} month streak' for _, r in flag2.iterrows()) or '- None identified'
flag3_text = '\n'.join(f'- `{r.admin1}`: {r.low_coverage_months} low-coverage months' for _, r in flag3.iterrows()) or '- None identified'

report_md = f'''# Phase 1 — Data Audit Report
**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}  
**Mode:** {"DEMO (synthetic data)" if DEMO_MODE else "REAL DATA"}  

## Dataset summary
| Field | Value |
|-------|-------|
| Raw rows | {len(df_raw):,} |
| Monthly panel rows | {len(monthly):,} |
| States | {monthly['admin1'].nunique()} |
| Date range | {metrics['date_range']['start']} → {metrics['date_range']['end']} |
| Target prevalence | {metrics['target_prevalence']:.1%} |
| Low-coverage cells | {metrics['low_coverage_fraction']:.1%} |

## Naive baseline metrics
| Metric | Value | Notes |
|--------|-------|-------|
| Brier score | {metrics['naive_baseline_M0']['brier_score']} | Lower is better; 0.25 = coin flip |
| Brier Skill Score | {metrics['naive_baseline_M0']['brier_skill_score']} | >0 means better than climatology |
| AUC-ROC | {metrics['naive_baseline_M0']['auc_roc']} | 0.5=random; 1.0=perfect |
| Expanding-window Brier | {metrics['naive_baseline_M0']['expanding_window_brier']} | PRIMARY benchmark for Phase 4 |

> **All Phase 4 models must achieve an expanding-window Brier score lower than the value above.**

## Coverage-bias flags

### ⚠️ Flag 1 — Silent week fraction > 30%
{flag1_text}

### ⚠️ Flag 2 — Long zero-fatality streak near a hotspot
{flag2_text}

### ⚠️ Flag 3 — Pilot state months with < 3 weeks observed
{flag3_text}

## Overdispersion test
See notebook cell 6. If `var/mean > 3` for fatalities, Poisson is ruled out.

## Limitations (to carry into Phase 2)
1. Admin1 centroids are state centroids, not event locations — sub-state mapping is misleading.
2. "Quiet" states near active conflict zones may be under-reported, not peaceful.
3. Weekly aggregation loses within-week sequencing; event ordering unknown.
4. ACLED may have sourcing gaps for rural Zamfara and Sokoto — UCDP cross-check needed.
'''

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_md)
print(f'Audit report saved: {report_path}')

---
## Phase 1 — Summary

### ✅ What we learned
1. **Data shape**: We have a weekly, admin1-level panel of events and fatalities. Aggregating to monthly gives a workable modelling unit.
2. **Overdispersion confirmed**: `var/mean >> 1` for both events and fatalities → negative-binomial is the right count model (not Poisson).
3. **Naive baseline is documented**: Every future model has a concrete Brier score to beat.
4. **Coverage gaps flagged**: Specific states and date ranges are marked as low-confidence and will carry uncertainty bands in all outputs.

### ⚠️ What could be wrong
- **Demo mode**: If running on synthetic data, the overdispersion ratio and coverage flags are artefacts of the simulation, not ground truth.
- **Event-level collapse**: Aggregating across `event_type` hides whether a state is quiet because it has no Battles (good) or no Protests (different meaning).
- **Centroid coordinates**: Any map produced from this data shows state centroids, not true event locations. Do not use for sub-state analysis until event-level data arrives.
- **ACLED coverage policy**: ACLED notes that Nigeria coverage improved significantly post-2016. Pre-2016 data should be used with extra caution.

### ➡️ Next step (Phase 2)
Build the ingestion pipeline: UCDP GED download and alignment to the monthly admin1 panel, with a source-reliability score per state-period.

**One question before we proceed:**  
Do you have your ACLED export file ready to drop into `data/raw/acled_nigeria_weekly_agg.csv`?  
If yes, re-run the notebook and check that `DEMO_MODE = False` in cell 0.
